# Multi-Agent Reinforcement Learning for Intra-Domain Traffic Engineering

**A topology-agnostic GNN policy, trained on 17 SNDlib backbones, evaluated zero-shot on Abilene, GÉANT and Germany50 with real measured operator traffic.**

MSc thesis — Verrell (`dean.ariefin.25@ucl.ac.uk`), UCL EEE.
This notebook is the single consolidated record of the project: the problem, the code, the
protocol, every reported number, and the results that *reversed* along the way.

---

## The question

Intra-domain routing in operator backbones is still shortest-path (OSPF/ECMP). It is
predictable and converges fast, but it is **traffic-oblivious**: it picks paths from the
topology alone and cannot move load off a hot link. Traffic engineering fixes this
centrally (MPLS-TE, an LP over the demand matrix), at the cost of a controller that must
know every demand.

> **Can a decentralised learned policy — one agent per router, acting on local link state —
> reduce the bottleneck load of a backbone it has never seen, without paying for it in
> packet loss or delay?**

## What is actually contributed

1. **A topology-agnostic policy.** One set of weights, trained on 17 SNDlib topologies
   (10–54 nodes), evaluated **zero-shot** on three held-out backbones with **real measured**
   traffic. No per-topology retraining. The routing survey identifies this as the gap that
   GDDR/GROM-style methods fail to close.
2. **A decentralised formulation with a GNN backbone.** Each node is an agent choosing the
   next hop from local neighbour utilisation plus a destination embedding; multi-hop
   message passing gives it a receptive field beyond its own links. Centralised training
   (MAPPO), decentralised execution.
3. **A matched-budget comparison against a centralised learned controller.** Every
   optimiser hyperparameter is identical; the single-agent baseline keeps Stable-Baselines3
   defaults and MAPPO moves to meet it, so the baseline is never handicapped.
4. **Packet-level validation.** All quality-of-service numbers are ns-3 FlowMonitor, not
   analytical. 558 simulations, 0 failures.
5. **A documented negative result and its diagnosis.** On Abilene the decentralised policy
   does *not* beat a correctly configured OSPF, and the reason — capacity-blind candidate
   paths — is isolated and measured rather than hidden.

## How to read this notebook

| Cell type | Meaning |
|---|---|
| Light code | Runs in seconds off committed artefacts (`results/*.json`, `topologies/*.json`, `logs/*.log`). Everything reported below is recomputed here, not transcribed. |
| `RUN_HEAVY = False` blocks | Training (~3 h/run × 20 runs) and ns-3 (558 sims). Commands are shown and are runnable; disabled by default. |

**Rule followed throughout:** every claim is stated with its counter-example. Sections
marked *"honest negative"* are not caveats bolted on at the end — they are results.

---
# 1. Environment and provenance

Everything below runs from the repository root with the project venv
(`~/thesis/ns3ai-venv`, Python 3.9) active.

In [ ]:
import json, os, re, sys, glob, subprocess, tarfile
from pathlib import Path
from itertools import islice

import numpy as np
import networkx as nx
import matplotlib
import matplotlib.pyplot as plt

ROOT = Path.home() / "thesis"
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

RESULTS = ROOT / "results"
TOPOS   = ROOT / "topologies"
LOGS    = ROOT / "logs"

RUN_HEAVY = False   # set True to re-run training / ns-3 (hours to days)

def jload(p):
    return json.loads(Path(p).read_text())

print("python    ", sys.version.split()[0])
print("numpy     ", np.__version__)
print("networkx  ", nx.__version__)
print("matplotlib", matplotlib.__version__)
try:
    import torch; print("torch     ", torch.__version__, "| cuda:", torch.cuda.is_available())
except Exception as e:
    print("torch      unavailable:", e)
print("root      ", ROOT)

In [ ]:
# Provenance: the exact tree these results came from.
def sh(cmd):
    try:
        return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()
    except Exception as e:
        return f"<{e}>"

print("branch :", sh("git rev-parse --abbrev-ref HEAD"))
print("commit :", sh("git rev-parse --short HEAD"))
print("date   :", sh("git log -1 --format=%ci"))
print()
print("Pinned externals")
print("  ns-3          3.42                      (~/thesis/ns-3-dev, gitignored)")
print("  ns3-ai        b8c9858  (main, NOT v1.2.0 tag)")
print("  ns3-ospf      c56950f  (vendor/ns3-ospf, ported to ns-3.42 -> ns3_patches/)")
print("  SNDlib        native .txt instances -> topologies/*_sndlib.json via sndlib_to_json.py")
print()
print("Frozen dependency set: requirements_freeze.txt "
      f"({len((ROOT/'requirements_freeze.txt').read_text().splitlines())} pins)")

### Repository map

| Path | Role |
|---|---|
| `marl_routing/topology.py` | topology JSON → NetworkX; the single source of truth shared with the ns-3 C++ side |
| `marl_routing/ospf_metric.py` | OSPF cost = refBW/cap; weighted Dijkstra, equal-cost sets, bottleneck utilisation |
| `marl_routing/real_traffic.py` | SNDlib measured demand archives, temporally split 70/30 |
| `marl_routing/tmgen_traffic.py` | TMgen modulated-gravity synthetic traffic for the 17 training topologies |
| `marl_routing/graph_routing_env.py` | **centralised** env: one step routes a whole demand over k=3 candidates |
| `marl_routing/topo_agnostic_marl_env.py` | **decentralised** env: one step is one hop, one agent per node |
| `marl_routing/marl_gnn.py` | GNN actor–critic + `GNNMAPPO` (custom MAPPO) |
| `train_single_tier2.py`, `train_marl_gnn_tier2.py` | the two trainers |
| `export_topoagn_routes.py`, `export_topoagn_marl_routes.py`, `export_ecmp_routes.py` | policy → explicit per-flow paths for ns-3 |
| `ns-3-dev/scratch/abilene-validate/` | packet-level scenario; installs per-flow static routes |
| `rebuild_ns3_grid.py`, `fill_offered_grid.py`, `decompose_reward.py`, `significance_test.py` | aggregation and statistics |
| `train_seeds10.sh`, `eval_seeds10.sh` | the launchers that produced the reported grid |

---
# 2. Networks

**Training:** 17 SNDlib topologies, 10–54 nodes, 21–80 undirected edges.
**Evaluation:** Abilene (12), GÉANT (22), Germany50 (50) — *never trained on*, and the only
three with publicly measured traffic.

The split is not arbitrary: the three evaluation backbones are excluded from training so
every reported number is zero-shot generalisation to an unseen graph.

In [ ]:
from marl_routing.topology import load as load_topology
from marl_routing import ospf_metric as om

TRAIN_TOPOS = ["atlanta_sndlib", "cost266_sndlib", "dfn-bwin_sndlib", "dfn-gwin_sndlib",
               "di-yuan_sndlib", "france_sndlib", "india35_sndlib", "janos-us_sndlib",
               "newyork_sndlib", "nobel-eu_sndlib", "nobel-germany_sndlib",
               "nobel-us_sndlib", "norway_sndlib", "pdh_sndlib", "polska_sndlib",
               "ta1_sndlib", "zib54_sndlib"]
TEST_TOPOS  = ["abilene_sndlib", "geant_sndlib", "germany50_sndlib"]

def topo_row(name):
    t = load_topology(name)
    G = t.graph
    caps = np.array([G[u][v]["capacity"] for u, v in G.edges()], float)
    und  = G.number_of_edges() // 2
    return dict(name=name.replace("_sndlib", ""), n=G.number_of_nodes(), m=und,
                deg=round(2 * und / G.number_of_nodes(), 2),
                diam=nx.diameter(G.to_undirected()),
                cap_gbps=f"{caps.min()/1000:g}-{caps.max()/1000:g}",
                distinct_caps=len(set(np.round(caps, 3))))

rows = [topo_row(n) for n in TEST_TOPOS + TRAIN_TOPOS]
hdr = f"{'topology':16}{'nodes':>6}{'edges':>7}{'avg deg':>9}{'diam':>6}{'cap (Gbps)':>14}{'distinct':>10}"
print("EVALUATION (zero-shot, real measured traffic)")
print(hdr); print("-" * len(hdr))
for r in rows[:3]:
    print(f"{r['name']:16}{r['n']:>6}{r['m']:>7}{r['deg']:>9}{r['diam']:>6}"
          f"{r['cap_gbps']:>14}{r['distinct_caps']:>10}")
print("\nTRAINING (17 topologies, TMgen synthetic traffic)")
print(hdr); print("-" * len(hdr))
for r in rows[3:]:
    print(f"{r['name']:16}{r['n']:>6}{r['m']:>7}{r['deg']:>9}{r['diam']:>6}"
          f"{r['cap_gbps']:>14}{r['distinct_caps']:>10}")
print(f"\ntraining set: {sum(r['n'] for r in rows[3:])} nodes total, "
      f"{min(r['n'] for r in rows[3:])}-{max(r['n'] for r in rows[3:])} per topology")

### The one heterogeneous link, and why it decides an entire chapter

GÉANT and Germany50 have **uniform** link capacities (40 Gbps). Abilene does not:
fourteen 9.92 Gbps links and **one 2.48 Gbps link**. That single link is responsible for the
largest result reversal in the project (§10), so it is worth seeing explicitly.

In [ ]:
t = load_topology("abilene_sndlib"); G = t.graph
caps = {}
for u, v in G.edges():
    caps.setdefault(round(G[u][v]["capacity"], 1), []).append((u, v))
for c, arcs in sorted(caps.items(), reverse=True):
    und = sorted({tuple(sorted(a)) for a in arcs})
    print(f"{c/1000:5.2f} Gbps  x{len(und):>2} undirected  "
          f"{'' if len(und) > 3 else und}")

W = om.weighted_graph(G)
slow = min(G.edges(), key=lambda e: G[e[0]][e[1]]["capacity"])
print(f"\nslow arc {slow}: capacity {G[slow[0]][slow[1]]['capacity']:.0f} Mbps, "
      f"OSPF cost {W[slow[0]][slow[1]]['w']:.1f} (fast links cost 1.0)")
print("-> a correctly configured OSPF routes AROUND it. A hop-count OSPF walks straight in.")

---
# 3. Traffic

Two sources, used for two different purposes.

**Evaluation — real measured traffic.** SNDlib ships demand-matrix archives for Abilene,
GÉANT and Germany50 (thousands of timestamped snapshots). The archive is split
**temporally**: first 70 % of the timeline is train-side, last 30 % is test. Reported
results use the test tail with evenly spaced timestamps, so matrix selection is a function
of *time*, not of a random seed, and is reproducible from the data alone.

**Training — TMgen modulated gravity.** No measured traffic exists for the 17 training
topologies, so demands are synthesised with the modulated-gravity model (TMgen), the
standard tool. Each topology gets 3 patterns × 5 volume scales `{0.6, 0.8, 1.0, 1.2, 1.5}`,
normalised so scale 1.0 puts the **OSPF bottleneck at exactly 100 %** — that normalisation
is what makes a single policy comparable across topologies whose absolute capacities differ
by an order of magnitude.

In [ ]:
from marl_routing.real_traffic import real_matrices

def pairs_of(topo):
    n = topo.graph.number_of_nodes()
    return [(a, b) for a in range(n) for b in range(n) if a != b]

t = load_topology("abilene_sndlib"); P = pairs_of(t)
mats = real_matrices("abilene_sndlib", P, load_scales=[1.0], n_per_scale=6, split="test")
print(f"{len(mats)} test matrices, each a vector over {len(P)} ordered pairs")
v = mats[0]
print(f"matrix 0: total demand {v.sum()/1000:8.1f} Gbps | active pairs "
      f"{int((v > 0).sum())}/{len(P)} | largest single demand {v.max():.1f} Mbps")
print(f"top-10 demands carry {100*np.sort(v)[-10:].sum()/v.sum():.1f}% of the total "
      "-- real traffic is heavily concentrated, which is exactly why a bottleneck exists")

In [ ]:
# Demand concentration across the three evaluation backbones.
fig, axes = plt.subplots(1, 3, figsize=(11, 3.1))
for ax, name in zip(axes, TEST_TOPOS):
    tt = load_topology(name); PP = pairs_of(tt)
    m = real_matrices(name, PP, load_scales=[1.0], n_per_scale=1, split="test")[0]
    n = tt.graph.number_of_nodes()
    D = np.zeros((n, n))
    for (a, b), r in zip(PP, m):
        D[a, b] = r
    im = ax.imshow(np.log10(D + 1), cmap="magma", aspect="auto")
    ax.set_title(f"{name.replace('_sndlib','')}  ({n} nodes)", fontsize=10)
    ax.set_xlabel("destination"); ax.set_ylabel("source" if ax is axes[0] else "")
    plt.colorbar(im, ax=ax, label="log10(Mbps+1)" if ax is axes[-1] else "")
fig.suptitle("Real measured demand matrices (one test snapshot each)", y=1.04)
fig.tight_layout(); plt.show()

---
# 4. Problem formulation

A topology is a directed graph $G=(V,E)$ with capacity $c_e$ on each arc. A demand matrix
gives a rate $d_{sd}$ for each ordered pair. A **routing** assigns each demand a path
$p_{sd}$. The arc load and the objective are

$$\ell_e \;=\; \sum_{(s,d)} d_{sd}\,\mathbb{1}[e \in p_{sd}], \qquad
U \;=\; \max_{e \in E} \frac{\ell_e}{c_e}\times 100\%.$$

$U$ is the **maximum offered load** on the bottleneck link. Three properties matter:

- It is the classical traffic-engineering objective (minimise the maximum link utilisation).
- It is **unbounded above**. $U = 150\%$ means the routing offers half again as much traffic
  to some link as that link can carry. This is *offered load*, not measured utilisation —
  a real link saturates at 100 % and converts the excess into loss. The two must never be
  presented as the same quantity, and are not, here.
- It is cheap. An ns-3 run of Germany50 takes ~17 minutes; $U$ takes microseconds. Training
  therefore optimises this **analytical surrogate**, and every *reported* quality-of-service
  number is re-measured in ns-3 afterwards.

### Regime labelling

Each test matrix is labelled by what **OSPF** would do with it:

$$\textbf{overload: } U_{\mathrm{OSPF}} \ge 100\% \qquad\qquad
  \textbf{feasible: } U_{\mathrm{OSPF}} < 100\%$$

The label is assigned by OSPF alone, before any learned policy is run, so it cannot be
gamed. ns-3 confirms the split cleanly: overload matrices lose 14–27 % of packets under
OSPF, feasible matrices lose 0.03–0.17 %.

### The reward

$$r \;=\; -\underbrace{\frac{U_{\pi}}{U_{\mathrm{OSPF}}}\times 100}_{\text{congestion}}
   \;-\; \underbrace{\beta \cdot \#\{\text{detour hops}\}}_{\text{delay}},\qquad \beta = 0.5$$

Normalising congestion by *this episode's* OSPF bottleneck is what makes one policy
trainable across 17 topologies at once: a raw bottleneck of 160 % on GÉANT and 100 % on
Abilene are not comparable rewards, but "fraction of what OSPF would have suffered" is.
OSPF scores exactly $-100$ by construction, which gives every episode a free reference
point. A **detour hop** is one that does not decrease OSPF-cost distance to the
destination; the stretch limit is $\sigma = 2$, capped at $\sigma_{\max} = 4$ cost units.

In [ ]:
# The surrogate, computed directly from its definition -- 8 lines, no framework.
def arc_index_of(G):
    arcs = list(G.edges())
    return arcs, {a: i for i, a in enumerate(arcs)}, \
           np.array([G[u][v]["capacity"] for u, v in arcs], float)

def bottleneck(G, paths, rates, arcs, ai, cap):
    load = np.zeros(len(arcs))
    for p, r in zip(paths, rates):
        if r <= 0 or p is None:
            continue
        for j in range(len(p) - 1):
            load[ai[(p[j], p[j + 1])]] += r
    return float((100.0 * load / cap).max())

t = load_topology("geant_sndlib"); G = t.graph
arcs, ai, cap = arc_index_of(G)
P = pairs_of(t)
W = om.weighted_graph(G)
m = real_matrices("geant_sndlib", P, load_scales=[5.0], n_per_scale=1, split="test")[0]
ospf_paths = [om.shortest_path(G, W, s, d, "weighted") for s, d in P]
U = bottleneck(G, ospf_paths, m, arcs, ai, cap)
print(f"GEANT, one test matrix at load scale 5.0")
print(f"  OSPF bottleneck offered load U = {U:.1f}%  ->  regime: "
      f"{'OVERLOAD' if U >= 100 else 'feasible'}")
print(f"  library agrees: {om.max_util(G, W, P, ai, cap, m, 'weighted'):.1f}%")

---
# 5. Baselines

### OSPF — and a defect that had to be fixed mid-project

Real OSPF costs a link $\mathrm{cost}(e) = \mathrm{refBW}/c_e$: a slow link is *expensive*
and gets routed around. This project's exporters originally used
`nx.shortest_path` with **no weight**, i.e. hop count.

On uniform-capacity topologies the two are identical, so GÉANT and Germany50 were never
affected. On Abilene they are not identical, and the difference is enormous. Every Abilene
result measured before the fix was scored against a straw man.

In [ ]:
t = load_topology("abilene_sndlib"); G = t.graph
arcs, ai, cap = arc_index_of(G); P = pairs_of(t); W = om.weighted_graph(G)
mats = real_matrices("abilene_sndlib", P, load_scales=[16.0], n_per_scale=6, split="test")

hop = [om.max_util(G, W, P, ai, cap, m, "hop")      for m in mats]
wgt = [om.max_util(G, W, P, ai, cap, m, "weighted") for m in mats]
print(f"Abilene, {len(mats)} real test matrices, bottleneck offered load (%)")
print(f"  hop-count OSPF (the defect) : {np.mean(hop):7.1f}   <- what was reported at first")
print(f"  weighted  OSPF (correct)    : {np.mean(wgt):7.1f}")
print(f"  the baseline was understated by {np.mean(hop) - np.mean(wgt):.1f} points\n")

# Mechanism: utilisation of the 2.48G link specifically.
slow_i = [ai[a] for a in arcs if G[a[0]][a[1]]["capacity"] < 5000]
for label, metric in [("hop-count OSPF", "hop"), ("weighted OSPF", "weighted")]:
    us = []
    for m in mats:
        load = np.zeros(len(arcs))
        for (s, d), r in zip(P, m):
            if r <= 0:
                continue
            p = om.shortest_path(G, W, s, d, metric)
            for j in range(len(p) - 1):
                load[ai[(p[j], p[j + 1])]] += r
        us.append((100 * load / cap)[slow_i].max())
    print(f"  {label:16} utilisation of the 2.48G link: {np.mean(us):6.1f}%")
print("\n-> weighted OSPF empties the slow link entirely; hop-count OSPF saturates it.")

### ECMP — and a second straw man

Real ECMP splits over paths of **equal OSPF cost**, not equal hop count, so it inherits
OSPF's capacity awareness. The original implementation split over equal-*hop* paths, which
made ECMP look worse than it is. After the fix, ECMP beats OSPF in both overload cells
(GÉANT 7.52 % loss vs 14.13 %; Germany50 22.90 % vs 27.29 %) — a materially stronger
baseline than the project reported for months.

ECMP also exists in two deliberately distinct flavours, and they are labelled separately
throughout:

| Flavour | Where | Behaviour |
|---|---|---|
| Fluid | analytical surrogate | each demand split *fractionally* over equal-cost paths |
| Flow-hashed | ns-3, `export_ecmp_routes.py` | one equal-cost path per flow, chosen by header hash — what routers actually do |

In [ ]:
t = load_topology("geant_sndlib"); G = t.graph
arcs, ai, cap = arc_index_of(G); P = pairs_of(t); W = om.weighted_graph(G)
m = real_matrices("geant_sndlib", P, load_scales=[5.0], n_per_scale=1, split="test")[0]

def ecmp_fluid(G, W, P, rates, ai, cap, metric="weighted"):
    load = np.zeros(len(cap))
    for (s, d), r in zip(P, rates):
        if r <= 0:
            continue
        ps = om.all_shortest_paths(G, W, s, d, metric)
        for p in ps:
            for j in range(len(p) - 1):
                load[ai[(p[j], p[j + 1])]] += r / len(ps)
    return float((100 * load / cap).max())

n_tied = [len(om.all_shortest_paths(G, W, s, d, "weighted")) for s, d in P]
print(f"GEANT: {100*np.mean(np.array(n_tied) > 1):.0f}% of pairs have >1 equal-cost path "
      f"(mean {np.mean(n_tied):.2f} paths)")
print(f"  OSPF bottleneck      {om.max_util(G, W, P, ai, cap, m, 'weighted'):6.1f}%")
print(f"  ECMP (fluid split)   {ecmp_fluid(G, W, P, m, ai, cap):6.1f}%")
print("  -> ECMP is a real improvement, not a formality. It is the baseline to beat.")

---
# 6. Two action spaces

The comparison in this thesis is **centralised vs decentralised at a matched budget**. The
two arms differ in exactly one structural thing: what a step is.

### Centralised — `GraphSeqRoutingEnv` (the single-agent baseline)

One step routes **one whole demand**. The controller sees the entire link-utilisation
vector and picks one of $k=3$ precomputed candidate paths. It is a strong, realistic
baseline for an SDN controller — and it is the arm this work must beat to justify
decentralisation.

- observation: per-candidate features (load on the candidate's bottleneck, hops, …) over the whole graph
- action: $\{1,2,3\}$
- reward: once, at the end of the demand

### Decentralised — `TopoAgnosticMARLEnv` (the proposed method)

One step is **one hop**. Every node is an agent; the agent at the current node picks a
neighbour from its own link loads and the destination. **No agent ever sees the whole path.**
Paths emerge from a chain of local decisions.

- observation: per-neighbour features (utilisation, whether it reduces cost-distance) + demand rate; degree-padded, **no node one-hots**, which is what makes the policy transferable to an unseen graph
- action: which neighbour, masked to live arcs and to the stretch budget
- reward: incrementally, per hop
- training: centralised critic over global arc state; execution: fully local

Loop-freedom and termination are enforced structurally by the stretch budget
($\sigma = 2$, $\sigma_{\max} = 4$ cost units accumulated), not learned.

In [ ]:
from marl_routing.gnn_routing_agent import compute_ksp

# The centralised action space, made concrete.
t = load_topology("geant_sndlib"); G = t.graph; W = om.weighted_graph(G)
s, d = 0, 15
for i, p in enumerate(compute_ksp(W, s, d, k=3, weight="w")):
    cost = sum(W[p[j]][p[j+1]]["w"] for j in range(len(p) - 1))
    print(f"  candidate {i+1}: {len(p)-1} hops, cost {cost:.1f}   {p}")
print("\nThe centralised policy picks one of these three. The decentralised policy is never")
print("shown them -- it reconstructs a path one neighbour at a time.")

### Capacity-blind candidates: the bug behind the Abilene negative result

The candidate paths were originally **hop**-shortest. On Abilene that means for many pairs
*all three* candidates traverse the 2.48 Gbps link — the policy is then structurally
incapable of avoiding it, no matter how well it is trained. The hop-based stretch limit
compounded this by forbidding the longer capacity-avoiding route.

The fix threads `--metric weighted` through the **environments**, not just the exporters:
candidate paths, distance-to-destination, the detour test and the stretch accounting are all
in OSPF-cost units. Here is the before/after that justified re-running everything.

In [ ]:
t = load_topology("abilene_sndlib"); G = t.graph; W = om.weighted_graph(G)
P = pairs_of(t)
slow_und = {tuple(sorted(a)) for a in G.edges() if G[a[0]][a[1]]["capacity"] < 5000}

def slow_hits(weight):
    hits = tot = 0
    for s, d in P:
        for p in compute_ksp(W if weight else G, s, d, k=3, weight=weight):
            tot += 1
            hits += any(tuple(sorted((p[j], p[j+1]))) in slow_und for j in range(len(p)-1))
    return hits, tot

for label, w in [("hop-shortest candidates (capacity-BLIND)", None),
                 ("cost-shortest candidates (capacity-AWARE)", "w")]:
    h, tot = slow_hits(w)
    print(f"  {label:44} slow link in {h:>3}/{tot} candidates ({100*h/tot:4.1f}%)")
print("\nSame policy weights, different candidate sets. Halving the exposure is what turned")
print("the Abilene delay penalty from 3-4x OSPF into parity (Section 10).")

In [ ]:
# Verification: on a uniform-capacity topology the fix changes nothing that matters.
for name in ["geant_sndlib", "germany50_sndlib"]:
    tt = load_topology(name); GG = tt.graph; WW = om.weighted_graph(GG)
    PP = pairs_of(tt)
    diff_path = diff_len = 0
    for s_, d_ in PP:
        a = om.shortest_path(GG, WW, s_, d_, "hop")
        b = om.shortest_path(GG, WW, s_, d_, "weighted")
        diff_path += (a != b)
        diff_len  += (len(a) != len(b))
    print(f"  {name.replace('_sndlib',''):12} {len(PP):>5} pairs | different path "
          f"{diff_path:>4} | different COST {diff_len:>4}")
print()
print("Every path the two metrics disagree on has IDENTICAL cost -- the disagreement is")
print("pure tie-breaking among equal-cost paths, which is what uniform capacity implies.")
print("The bottleneck can still move a little as a result (a different tie can load a")
print("different link), so the metric change was re-measured on GEANT and Germany50 rather")
print("than assumed to be a no-op. Only on Abilene does it change routing DECISIONS.")


---
# 7. The policies

### Decentralised: GNN actor–critic trained with MAPPO

The actor runs **3 rounds of message passing** over the topology before acting. This is the
single change that turned the topology-agnostic MARL result from negative to positive: with
a purely local (0-round) actor the agent cannot tell a link that is congested *because of
what is downstream* from one that is merely busy, and it oscillates. Three rounds give it a
3-hop receptive field while keeping execution local — a router exchanges messages with its
neighbours, which is exactly what a routing protocol already does.

The critic is centralised (it sees the padded global arc-utilisation vector) and is used
**only during training**. At execution, each node needs its own neighbourhood and the
destination. Nothing else.

### Centralised: single-agent GNN + PPO (Stable-Baselines3)

Same GNN feature extractor, same 3 rounds, but it consumes the whole graph and emits a
choice among $k=3$ whole paths. Stable-Baselines3 PPO with **untouched defaults** — see §8
for why that direction of matching is deliberate.

In [ ]:
import torch
from marl_routing.marl_gnn import GNNActorCritic
from marl_routing.topo_agnostic_marl_env import TopoAgnosticMARLEnv as MEnv

def count(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

for h in (32, 64):
    net = GNNActorCritic(MEnv.node_f_dim, MEnv.gstate_dim, hidden=h, rounds=3,
                         act_dim=MEnv.max_deg, local_f=MEnv.nf_dim)
    print(f"  MARL GNN actor-critic  hidden={h:<3} rounds=3  ->  {count(net):>8,} trainable parameters")
print(f"  Single-agent GNN + PPO hidden=64  rounds=3  ->  {100740:>8,} trainable parameters (measured)")
print("\nThe decentralised policy is ~5.7x SMALLER than the centralised baseline it is")
print("compared against. Width was chosen on held-out TRAINING topologies (Section 9),")
print("never on the three evaluation backbones.")

### Why the reward form had to be unified (a late correction worth recording)

The two arms originally used delay penalties that shared a value ($\beta = 0.5$) but not a
*meaning*: the centralised arm charged $\beta$ per **extra hop** and normalised it together
with the congestion term; the decentralised arm charged a flat $\beta$ per **detour hop**,
unnormalised. The measured consequence of the centralised form was severe — the
cost-shortest candidate was myopically optimal for **95.2 %** of GÉANT and **97.2 %** of
Germany50 demands, and the policy simply reproduced OSPF on 90.3 % of GÉANT demands.

Re-running the centralised arm under the decentralised reward form (`--reward-form marl`)
changed its results substantially, and *in its favour*:

| Cell | old form | new form |
|---|---|---|
| Abilene feasible loss | 3.69 % | **0.24 %** |
| Abilene feasible delay | 37.84 ms | **13.77 ms** |
| Abilene overload loss | 21.64 % | **13.41 %** (best in cell) |
| Germany50 feasible util | 74.8 % | **61.5 %** |
| Germany50 overload loss | 16.73 % | 18.17 % *(the one regression)* |

The reported centralised arm is the corrected one (`_singleH64gRM`). **This strengthened the
baseline**, which is the point of doing it. At the three seeds available then it flipped
the six-cell head-to-head from 5–1 to an even 3–3. Extending to ten seeds moved it back to
**5–1 for the decentralised arm** — the centralised policy now wins only Abilene feasible,
and there only on utilisation (91.4 % vs 92.4 %), while losing that same cell on packet
loss (1.01 % vs 0.56 %). Both readings are recorded because the reversal of a reversal is
exactly the kind of thing three seeds cannot settle.

**Not tested, stated as a limitation:** whether the decentralised policy would *also*
improve under the centralised reward form. Only the baseline was re-run.

---
# 8. Training protocol

### Matched budget, and the direction of matching

Four PPO hyperparameters differed between the arms ($\gamma$, epochs, minibatch, buffer).
They were matched — **by moving MAPPO onto Stable-Baselines3's defaults**, not by trimming
the baseline. Handicapping a baseline to match one's own method is the classic way to
manufacture a win, and it was avoided deliberately.

In [ ]:
HYPER = [
    ("Environment steps",        "1.5e6",  "1.5e6"),
    ("Rollout buffer",           "2048",   "2048 (8x256)"),
    ("Policy updates",           "732",    "732"),
    ("Learning rate",            "3e-4",   "3e-4"),
    ("Discount gamma",           "0.995",  "0.995"),
    ("GAE lambda",               "0.95",   "0.95"),
    ("Clip range",               "0.2",    "0.2"),
    ("Epochs per update",        "10",     "10"),
    ("Minibatch",                "256",    "256"),
    ("Entropy coefficient",      "0.01",   "0.01"),
    ("Value coefficient",        "0.5",    "0.5"),
    ("Gradient-norm clip",       "0.5",    "0.5"),
    ("Message-passing rounds",   "3",      "3"),
    ("Hidden width",             "32",     "64"),
    ("Trainable parameters",     "17,570", "100,740"),
    ("Candidate paths k",        "-- (hop-by-hop)", "3"),
    ("Seeds",                    "0..9",   "0..9"),
]
print(f"{'':28}{'MARL (MAPPO)':>18}{'Single-agent (PPO)':>22}")
print("-" * 68)
for k, a, b in HYPER:
    mark = "" if a == b else "   <- differs by design"
    print(f"{k:28}{a:>18}{b:>22}{mark}")
print("\nShared environment settings: delay penalty beta=0.5, detour allowance sigma=2,")
print("max stretch 4 cost units, reward normalised by the episode's OSPF bottleneck,")
print("500-demand cap on training matrices, metric=weighted, identical 17 training topologies.")

### The budget mismatch that cannot be fixed, and which way it cuts

One centralised step routes a **whole demand**; one decentralised step routes **one hop**.
Average path length is ~6 hops, so at 1.5 M environment steps the centralised controller
processes roughly **6× more demands**. The budget is matched on environment steps and
unmatched on routing work.

This is stated as a limitation, and the direction matters: **the mismatch favours the
centralised baseline**. The decentralised policy winning anyway cannot be an
under-training artefact of the baseline.

In [ ]:
print("Training launcher (train_seeds10.sh) -- the exact commands behind every reported number:\n")
print('''  # centralised baseline: SB3 defaults untouched, unified reward form
  python train_single_tier2.py --seed $s --timesteps 1500000 --traffic tmgen \\
      --k-paths 3 --hidden 64 --rounds 3 --n-envs 8 --n-steps 256 --ent-coef 0.01 \\
      --metric weighted --reward-form marl --tag _singleH64gRM

  # decentralised: moved onto the baseline's PPO settings
  python train_marl_gnn_tier2.py --seed $s --updates 732 --rollout 2048 \\
      --gamma 0.995 --n-epochs 10 --minibatch 256 --hidden 32 --rounds 3 \\
      --traffic tmgen --metric weighted --tag _tier2m15cm
''')
print("20 runs (2 arms x 10 seeds), OMP_NUM_THREADS=1, nice -n 19, 14 concurrent.")
print("Wall clock on the EPYC 7543: ~3 h per centralised run, ~1.3 h per decentralised run.")

if RUN_HEAVY:
    subprocess.run(["bash", "train_seeds10.sh"], check=True)
else:
    print("\n[RUN_HEAVY=False] not re-running. Existing artefacts:")
    for pat, lbl in [("single_singleH64gRM_seed*", "centralised"),
                     ("marlgnn_tier2m15cm_seed*", "decentralised")]:
        print(f"  {lbl:15} {len(sorted(RESULTS.glob(pat)))} seed directories")

### Committing to all ten seeds before looking at them

The reported results moved from 3 seeds to 10 for a specific reason: at 3 seeds the
feasible-regime loss on Abilene was a mean over a **bimodal** set — seeds 0 and 1 matched
OSPF (0.16 %, 0.34 %) while seed 2 failed (2.42 %). Three points cannot distinguish "one
unlucky run" from "this happens a third of the time".

Ten seeds were launched with a prior commitment to report **all** of them. Running extra
seeds and keeping the flattering subset would have invalidated the very variance claim the
extension was meant to settle.

In [ ]:
# Training curves, straight from the logs.
runs = []
for f in sorted(LOGS.glob("train_marlh32cm_s*.log"),
                key=lambda p: int(re.search(r"_s(\d+)", p.name).group(1))):
    upd, ret = [], []
    for line in f.read_text().splitlines():
        m = re.search(r"upd\s+(\d+)/\d+\s+ep_ret~(-?[\d.]+)", line)
        if m:
            upd.append(int(m.group(1))); ret.append(float(m.group(2)))
    if upd:
        runs.append((np.array(upd), np.array(ret), f.name))

print(f"{len(runs)} MARL training logs found\n")
if runs:
    grid  = runs[0][0]
    stack = np.vstack([np.interp(grid, u, r) for u, r, _ in runs])
    W_ = 15
    def smooth(y):
        pad = np.r_[np.full(W_//2, y[0]), y, np.full(W_//2, y[-1])]
        return np.convolve(pad, np.ones(W_)/W_, mode="valid")[:len(y)]
    sm = np.vstack([smooth(stack[i]) for i in range(len(stack))])

    fig, ax = plt.subplots(figsize=(7, 3.4))
    ax.plot(grid, stack.mean(0), color="#3B49B8", lw=0.7, alpha=0.22, label="seed mean (raw)")
    ax.fill_between(grid, sm.min(0), sm.max(0), color="#3B49B8", alpha=0.16, lw=0,
                    label="seed range (smoothed)")
    ax.plot(grid, sm.mean(0), color="#3B49B8", lw=2, label="seed mean (smoothed)")
    ax.axhline(-100, color="#B3352B", ls="--", lw=1,
               label="congestion-only OSPF reference (-100)")
    ax.set_xlabel("policy update"); ax.set_ylabel("mean episode return")
    ax.set_title("MARL (h=32) training return, 17 training topologies")
    ax.legend(frameon=False, fontsize=8); ax.grid(alpha=.3)
    fig.tight_layout(); plt.show()
    print(f"final smoothed seed mean: {sm.mean(0)[-1]:.1f}")
    print("\nNOTE the -100 line is NOT a pass mark for this curve. OSPF scores exactly -100 on")
    print("the CONGESTION term alone; the plotted return also carries the UNNORMALISED delay")
    print("penalty, and these are the 17 TRAINING topologies (several of them dense, with")
    print("many cheap detours to be charged for). The curve is evidence of convergence, not")
    print("of beating OSPF -- that comparison is made on the held-out backbones in Sections")
    print("10-11, on the bottleneck objective rather than on the scalarised return.")
    print("\nThe raw trace is noisy because each episode samples ONE of 17 topologies whose")
    print("bottlenecks differ by an order of magnitude -- the smoothing removes that sampling")
    print("noise, it is not cosmetic.")

---
# 9. Model selection, done on the training set

The hidden width $h \in \{32, 64\}$ was selected on **held-out TMgen matrices from the 17
training topologies**, generated with a seed distinct from the training seed. The three
evaluation backbones were not consulted. This matters: with only three test networks,
selecting width on them would make every subsequent number a selection artefact.

In [ ]:
ws = jload(RESULTS / "width_selection.json")
print(f"Selection set: {len(ws['topos'])} training topologies, held-out matrices "
      f"(gen_seed={ws['gen_seed']})\n")
print(f"{'arm':6}{'mean improvement vs OSPF (pt)':>32}{'sd over seeds':>16}")
for arm, rec in ws["per_arm"].items():
    print(f"{arm:6}{rec['mean_delta_pt']:>32.2f}{rec['std_delta_pt']:>16.2f}")
best = max(ws["per_arm"], key=lambda a: ws["per_arm"][a]["mean_delta_pt"])
print(f"\nselected: {best}  (larger reduction of the bottleneck AND the smaller model)")
print("h=64 was kept only as a sensitivity arm and is not the reported result.")

---
# 10. Results I — bottleneck offered load (analytical)

Nine test matrices per regime, three from each backbone, all real measured traffic. Learned
arms are mean ± sd **across model seeds**, each seed contributing its own mean over
matrices.

In [ ]:
off = jload(RESULTS / "offered_grid.json")
ARMS = [("ospf", "OSPF"), ("ecmp", "ECMP"), ("single", "Single-agent"), ("marlh32", "MARL h=32")]
CELLS = [f"{t}/{r}" for r in ("overload", "feasible")
         for t in ("abilene", "geant", "germany50")]

print("Maximum offered load on the bottleneck link (%), lower is better")
print(f"{'cell':22}" + "".join(f"{lbl:>18}" for _, lbl in ARMS))
print("-" * (22 + 18 * len(ARMS)))
for c in CELLS:
    if c not in off:
        continue
    line = f"{c:22}"
    vals = {a: off[c][a]["mean"] for a, _ in ARMS if a in off[c]}
    best = min(vals, key=vals.get)
    for a, _ in ARMS:
        r = off[c].get(a)
        if r is None:
            line += f"{'--':>18}"; continue
        cell = f"{r['mean']:.1f}+-{r['sd']:.1f}" if r["seeds"] > 1 else f"{r['mean']:.1f}"
        line += f"{('*' + cell) if a == best else cell:>18}"
    print(line)
    if c.endswith("germany50/overload"):
        print()
print("\n* = best in cell.  100% is the capacity line: above it the routing offers more")
print("traffic to some link than the link can carry.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
colors = {"ospf": "#37424C", "ecmp": "#14A8A0", "single": "#D9730D", "marlh32": "#3B49B8"}
topos  = ["abilene", "geant", "germany50"]
for ax, regime in zip(axes, ["overload", "feasible"]):
    wdt = 0.8 / len(ARMS)
    for k, (a, lbl) in enumerate(ARMS):
        xs, ys, es = [], [], []
        for i, tp in enumerate(topos):
            r = off.get(f"{tp}/{regime}", {}).get(a)
            if r:
                xs.append(i - 0.4 + wdt * (k + .5)); ys.append(r["mean"]); es.append(r["sd"])
        ax.bar(xs, ys, wdt * .88, yerr=es, capsize=2, color=colors[a], label=lbl,
               edgecolor="white", lw=.6, error_kw=dict(lw=.8))
    if regime == "overload":
        ax.axhline(100, color="#B3352B", ls="--", lw=1)
        ax.text(2.45, 103, "capacity", color="#B3352B", ha="right", fontsize=8)
    ax.set_xticks(range(3)); ax.set_xticklabels(["Abilene", "GÉANT", "Germany50"])
    ax.set_title(f"{regime} regime"); ax.grid(axis="y", alpha=.3); ax.set_axisbelow(True)
axes[0].set_ylabel("max offered load (%)")
axes[0].legend(frameon=False, fontsize=8, ncol=2)
fig.tight_layout(); plt.show()

### Reading the table honestly

**What holds.** The decentralised policy is the best arm on GÉANT and Germany50 in both
regimes — 97.7 % on GÉANT overload (i.e. it pulls a 145 % OSPF bottleneck back *below
capacity*), and 49.0 % / 38.9 % in the feasible cells, roughly a 30-point reduction against
OSPF. It also has by far the tightest seed spread.

**What does not.** On **Abilene the decentralised policy does not beat OSPF** — 125.4 %
against OSPF's 129.3 % in overload is within seed noise, and in the feasible cell it is
beaten by both OSPF (57.2) and the centralised policy (55.6). Against a correctly weighted
OSPF, 12 nodes with one slow link is simply not a network where hop-by-hop decentralised
routing has anything to add: OSPF's cost metric already solves the only real decision in the
graph.

The honest framing is **"learned routing helps where there is routing freedom to exploit,
and scale is what creates that freedom"** — not "MARL beats OSPF".

### Seed stability — the clearest architectural result

Averages hide the property an operator would actually care about: *will the next training
run behave like this one?*

In [ ]:
print("Standard deviation across 10 model seeds (percentage points of offered load)")
print(f"{'cell':22}{'Single-agent':>16}{'MARL h=32':>14}{'ratio':>10}")
print("-" * 62)
for c in CELLS:
    r = off.get(c, {})
    if "single" in r and "marlh32" in r:
        s, m = r["single"]["sd"], r["marlh32"]["sd"]
        print(f"{c:22}{s:>16.1f}{m:>14.1f}{(s/m if m else float('nan')):>10.1f}x")
print("\nThe decentralised policy is the tighter arm in EVERY cell, under IDENTICAL PPO")
print("hyperparameters -- so the centralised arm's fragility is architectural, not a")
print("settings artefact. At three seeds Abilene was a counter-example (the centralised arm")
print("looked tighter there); it did not survive the extension to ten, which is the whole")
print("reason the extension was run.")

### Reference points: what a clairvoyant heuristic achieves

Neither learned policy should be credited with what a simple greedy heuristic on the same
action space already gets. Both greedy references require global link state after every
decision, and the path-level one requires the demand matrix in advance, so **neither is
deployable**. They are bounds, not competitors.

In [ ]:
gk = jload(RESULTS / "greedy_k3_grid.json")
print(f"{'cell':22}{'OSPF':>10}{'greedy k=3 (path)':>20}{'MARL h=32':>14}")
print("-" * 66)
for c in CELLS:
    if c in gk and c in off:
        print(f"{c:22}{off[c]['ospf']['mean']:>10.1f}{gk[c]['greedy_k3']:>20.1f}"
              f"{off[c]['marlh32']['mean']:>14.1f}")
print('''
Two things separate sharply:

  Acting greedily over WHOLE PATHS is very strong -- best in every cell. At 50 nodes the
  learned policy essentially reaches it (117.0 vs 116.4 in overload); at 12 and 22 nodes a
  clairvoyant heuristic still does better. This bounds the contribution honestly.

  Acting greedily HOP BY HOP -- the decentralised action space -- is catastrophic: worse
  than OSPF on four of six cells, and 403.8% on Abilene overload against OSPF's 129.3%.
  Choosing the locally least-congested next hop walks demands into long paths that load
  links no single decision was accountable for. That the learned policy, acting in exactly
  that space, instead reaches 124.9% and 96.6% is the clearest evidence here that it has
  learned something beyond greed.''')

### What the policy trades away to get there

The reward has two terms. Decomposing the achieved return shows *how* the bottleneck was
reduced — and that the answer differs qualitatively with scale.

In [ ]:
rd = jload(RESULTS / "reward_decomp.json")
print(f"{'topology':12}{'return':>10}{'congestion':>13}{'delay':>9}"
      f"{'detour hops':>14}{'detour %':>11}{'util vs OSPF':>15}")
print("-" * 84)
for t_, r in rd.items():
    print(f"{t_:12}{r['return']:>10.1f}{r['congestion']:>13.1f}{r['delay']:>9.1f}"
          f"{r['detour_hops']:>14.1f}{r['detour_pct']:>10.1f}%{r['util_ratio']:>15.2f}")
print('''
Abilene and GEANT: the congestion term dominates and the delay penalty is small -- 3.0% and
4.4% of hops are detours, buying bottlenecks at 100% and 69% of OSPF's.

Germany50: the largest congestion win (48% of OSPF's bottleneck) but the delay penalty
(-58.7) is now the LARGER term, bought with detours on 13.5% of hops. The total return falls
below the -100 that OSPF scores by construction.

So on the largest topology the policy pays more in detours than it recovers in congestion
UNDER THIS REWARD. It is still the better router by the bottleneck objective; it is not
better by its own scalarised return. Both are true and both are reported.''')

---
# 11. Results II — packet-level quality of service (ns-3)

The analytical surrogate optimises a bottleneck number. It cannot say whether the resulting
paths actually deliver packets. Every reported loss/delay/utilisation figure below is
**ns-3 FlowMonitor**, from 558 simulations with 0 failures.

### How a learned policy gets into a packet simulator

```
  policy (.pt / .zip)
        │  export_topoagn_marl_routes.py --metric weighted
        ▼
  routing_seedN.json          explicit node-list path for every flow
        │
        ▼
  ns-3  scratch/abilene-validate    installs PER-FLOW STATIC HOST ROUTES
        │                            UDP OnOff, 1400 B, DropTail(100), no AQM, no TCP
        ▼
  ns3_gnn_N.json              FlowMonitor: loss %, mean delay, per-link utilisation
```

**Deployment caveat, stated in the thesis rather than buried:** those per-flow routes are
effectively source-routed. Plain destination-based IP forwarding **cannot express them** —
deploying this policy would require MPLS or segment routing. This is a real limitation of
the approach, not of the simulation.

### Simulation configuration

- 9 s simulated, flows active 2 s → end, utilisation accumulated over an 8 s window and
  corrected by 7/8 for the active fraction
- demand rates **and** link capacities both divided by 20 — preserves every utilisation,
  loss and delay *ratio* while keeping packet counts tractable
- propagation delay at nanosecond resolution (necessary: several Germany50 links are < 1 ms)
- one-way UDP only, so loss and delay follow from routing rather than transport feedback
- all loss is buffer overflow: DropTail 100 packets, no AQM, no shaping

**OSPF is not re-simulated per seed.** It is deterministic given the matrix, and this was
verified (`ns3_ospf_{0,8,11}.json` agree to four decimals across seeds). Re-running it would
have added 126 simulations for bit-identical output.

In [ ]:
if RUN_HEAVY:
    subprocess.run(["bash", "eval_seeds10.sh"], check=True)
    subprocess.run([sys.executable, "rebuild_ns3_grid.py"], check=True)
else:
    dirs = sorted(RESULTS.glob("ns3f_*"))
    sims = sum(len(list(d.glob("ns3_*.json"))) for d in dirs)
    print(f"[RUN_HEAVY=False] using committed artefacts: {len(dirs)} export dirs, "
          f"{sims} simulation state files")
    print("\nRe-run with:  bash eval_seeds10.sh && python rebuild_ns3_grid.py")

In [ ]:
ns3 = jload(RESULTS / "final_ns3_grid.json")

def grid(field, sd, title, lower_better=True):
    print(f"\n{title}")
    print(f"{'cell':22}" + "".join(f"{l:>18}" for _, l in ARMS))
    print("-" * (22 + 18 * len(ARMS)))
    for c in CELLS:
        if c not in ns3:
            continue
        vals = {a: ns3[c][a][field] for a, _ in ARMS if a in ns3[c]}
        best = (min if lower_better else max)(vals, key=vals.get)
        line = f"{c:22}"
        for a, _ in ARMS:
            r = ns3[c].get(a)
            if r is None:
                line += f"{'--':>18}"; continue
            s = f"{r[field]:.2f}+-{r[sd]:.2f}" if r["seeds"] > 1 else f"{r[field]:.2f}"
            line += f"{('*' + s) if a == best else s:>18}"
        print(line)

grid("loss_pct",    "loss_sd",    "PACKET LOSS (%)  -- the metric that matters in overload")
grid("delay_ms",    "delay_sd",   "MEAN END-TO-END DELAY (ms)")
grid("maxutil_pct", "maxutil_sd", "MAX LINK UTILISATION (%)  -- the metric that matters when feasible")
print("\n* = best in cell.  Utilisation SATURATES at 100% in ns-3: excess offered load")
print("becomes loss, so read utilisation in the feasible regime and loss in overload.")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 6.4))
panels = [("loss_pct", "loss_sd", "packet loss (%)", "overload"),
          ("loss_pct", "loss_sd", "packet loss (%)", "feasible"),
          ("maxutil_pct", "maxutil_sd", "max link utilisation (%)", "overload"),
          ("maxutil_pct", "maxutil_sd", "max link utilisation (%)", "feasible")]
for ax, (fld, sdf, ylab, regime) in zip(axes.ravel(), panels):
    wdt = 0.8 / len(ARMS)
    for k, (a, lbl) in enumerate(ARMS):
        xs, ys, es = [], [], []
        for i, tp in enumerate(topos):
            r = ns3.get(f"{tp}/{regime}", {}).get(a)
            if r:
                xs.append(i - 0.4 + wdt * (k + .5)); ys.append(r[fld]); es.append(r[sdf])
        ax.bar(xs, ys, wdt * .88, yerr=es, capsize=2, color=colors[a], label=lbl,
               edgecolor="white", lw=.6, error_kw=dict(lw=.8))
    ax.set_xticks(range(3)); ax.set_xticklabels(["Abilene", "GÉANT", "Germany50"])
    ax.set_ylabel(ylab); ax.set_title(f"{regime} regime", fontsize=10)
    ax.grid(axis="y", alpha=.3); ax.set_axisbelow(True)
axes[0][0].legend(frameon=False, fontsize=8, ncol=2)
fig.suptitle("ns-3 packet-level results, mean ± sd over 10 model seeds", y=1.01)
fig.tight_layout(); plt.show()

### What the packet level says

**Overload.** The decentralised policy leads on **five of the six** loss and delay
measurements: lowest loss and delay on GÉANT (3.36 %, 18.1 ms), lowest loss and delay on
Germany50 (10.63 %, 17.1 ms), lowest delay on Abilene (112.8 ms). The single exception is
**Abilene packet loss, where ECMP and OSPF are best (14.31 %, 14.35 %) and the learned
policy trails at 14.90 %** — the same negative result as the analytical table, confirmed
independently at the packet level.

**Feasible.** Almost nothing separates the methods: loss ranges over 0.03–1.09 % and delay
over 1.8–21.6 ms for *every* method. **When the network is not congested, shortest-path
routing is already good enough**, and the honest conclusion is that learned routing has
nothing to offer there.

What the learned policy does buy in the feasible regime is **headroom**: 66.3 % max
utilisation on GÉANT against OSPF's 97.3 %, at *lower* loss (0.12 % vs 0.16 %) and *lower*
delay (8.5 ms vs 9.46 ms). Thirty points of spare capacity, free.

**And where headroom is not free.** On Germany50 feasible the decentralised policy reaches
**65.5 %** utilisation — 30 points below OSPF — but pays **1.09 %** packet loss, while the
centralised policy sits at 69.6 % and holds OSPF's own **0.03 %**. The last four points of
headroom are **bought with loss**, and that is reported as a trade rather than as a win.
It is also the cell with the widest seed spread of any (±16.1), so it is the least settled
number in the grid.

---
# 12. Statistical testing

Comparisons are **paired on the traffic matrix**: each learned routing is simulated on
exactly the same matrix as the OSPF routing it is compared against, so only the routing
varies. Matrices differ enormously in offered load, and an unpaired test would drown the
routing effect in between-matrix variance.

The 10 model seeds are **averaged before pairing** — they are ten draws of one policy, not
ten independent observations of a matrix, and treating them as separate samples would
inflate $n$. Nine matrices per regime, so the smallest attainable two-sided Wilcoxon
$p$ is 0.004.

In [ ]:
from scipy import stats

ARM_DIR = "marlh32"
DIRS = {"abilene": "Abilene", "geant": "GEANT", "germany50": "Germany50", "g50feas": "Germany50"}
SEEDS = range(10)

def read(stem, seed, kind, m):
    p = RESULTS / f"ns3f_{ARM_DIR}_{stem}_s{seed}" / f"ns3_{kind}_{m}.json"
    return jload(p) if p.exists() else None

rows = []
for stem, topo in DIRS.items():
    mats = sorted(int(re.search(r"ospf_(\d+)", f).group(1))
                  for f in glob.glob(str(RESULTS / f"ns3f_{ARM_DIR}_{stem}_s0" / "ns3_ospf_*.json")))
    for m in mats:
        o = read(stem, 0, "ospf", m)
        g = [x for x in (read(stem, s, "gnn", m) for s in SEEDS) if x is not None]
        if o is None or not g:
            continue
        rows.append(("overload" if o["loss_pct"] > 1.0 else "feasible", topo, m,
                     o["loss_pct"], float(np.mean([x["loss_pct"] for x in g])),
                     max(o["link_utils"]),
                     float(np.mean([max(x["link_utils"]) for x in g]))))

for regime, metric, oi, mi in [("overload", "packet loss (%)", 3, 4),
                               ("feasible", "max link utilisation (%)", 5, 6)]:
    sel = [r for r in rows if r[0] == regime]
    if not sel:
        continue
    o = np.array([r[oi] for r in sel]); g = np.array([r[mi] for r in sel]); d = g - o
    w = stats.wilcoxon(g, o); tt = stats.ttest_rel(g, o)
    print(f"=== {regime}: {metric}, MARL h=32 vs OSPF (n={len(sel)} paired matrices) ===")
    print(f"  OSPF {o.mean():7.2f}   MARL {g.mean():7.2f}   mean diff {d.mean():+.2f}   "
          f"median diff {np.median(d):+.2f}")
    print(f"  MARL better on {(d < 0).sum()}/{len(d)} matrices")
    print(f"  Wilcoxon signed-rank  W={w.statistic:.1f}, p={w.pvalue:.4f}")
    print(f"  paired t-test         t={tt.statistic:.2f}, p={tt.pvalue:.4f}")
    sub = [r for r in sel if r[1] != "Abilene"]
    if sub:
        o2 = np.array([r[oi] for r in sub]); g2 = np.array([r[mi] for r in sub])
        w2 = stats.wilcoxon(g2, o2)
        print(f"  excluding Abilene (n={len(sub)}): better on {(g2-o2 < 0).sum()}/{len(sub)}, "
              f"W={w2.statistic:.1f}, p={w2.pvalue:.4f}")
    print()

**Reading the tests.** In the feasible regime the decentralised policy lowers maximum link
utilisation on **all nine** paired matrices, by 19.1 points on average
($W=0$, $p=0.004$ — the smallest value attainable at this $n$). In overload it lowers packet
loss on **seven of nine**, by 9.0 points, reaching the conventional threshold
($W=5$, $p=0.039$); **the two matrices that favour OSPF are both Abilene matrices**.
Restricted to GÉANT and Germany50 the improvement holds on all six ($W=0$, $p=0.031$).

The Abilene-excluded test is quoted **alongside** the pooled one, never instead of it.

So: the headroom result is significant across every network tested; the loss result is
significant only where the decentralised formulation works at all.

---
# 13. Honest negatives, and results that reversed

Several findings in this project reversed under more data or a corrected baseline. They are
recorded because the pattern — *single-seed, short-budget or straw-man results are
provisional* — is itself a result.

| # | Claimed | What happened | Status |
|---|---|---|---|
| 1 | "Abilene: 7.19 % → 0.62 % bottleneck, a 10× win" | Baseline was **hop-count** OSPF. Correctly weighted OSPF scores 57.25 % and beats every learned arm by ~13 pt. Abilene has **no overload regime at all** under it. | **Reversed** |
| 2 | "The detours cost delay without buying congestion relief" (Abilene, 3–4× worse delay) | An artefact of **capacity-blind candidate paths**. With cost-shortest candidates the same weights give 11.4–12.1 ms vs OSPF's 11.45. | **Reversed** |
| 3 | "ECMP is worse than OSPF on Germany50" | That was equal-**hop** ECMP, a straw man. Real equal-**cost** ECMP beats OSPF in both overload cells. | **Reversed** |
| 4 | "MARL dominates the centralised baseline 5–1" | The centralised arm's reward form was inconsistent. Fixed, it wins Abilene ×2 and Germany50 feasible → an even **3–3**. | **Weakened** |
| 5 | "h=64 is the headline arm" | h=32 is better on GÉANT and Abilene with ~1/3 the seed spread; h=64 wins only Germany50. | **Corrected** |
| 6 | "MARL is the stable arm, the single agent is seed-fragile" | Held and strengthened at 10 seeds — MARL is now tighter in **all six** cells (Germany50 overload ±35.7 vs ±9.6). The Abilene counter-example seen at 3 seeds did not survive. | **Held** |
| 7 | "Zero-shot works better with more topological diversity" (245 Topology Zoo graphs) | It did not. Germany50 degraded out of distribution. Kept as a negative. | **Negative, kept** |

### The Abilene failure, diagnosed rather than excused

Mechanism, measured (mean utilisation of the 2.48 Gbps link across the full
18-matrix evaluation set, each at its own load):

| Routing | slow-link utilisation |
|---|---|
| hop-count OSPF | 83.6 % |
| **weighted OSPF** | **0.0 %** — avoids it entirely |
| MARL h=32/64 | 58.0 % — learned to *partially* relieve it, cannot eliminate it |

The policy cannot eliminate it because for some pairs **all three** candidates traverse the
slow link. Raising the load so weighted OSPF is genuinely congested (129.3 % on its overload
subset) does not rescue the learned arms either — they sit at 148.9 % and worse. The failure
is **structural**, not a load-distribution artefact.

The general lesson is worth more than the Abilene number: **on heterogeneous-capacity
topologies a capacity-aware baseline beats capacity-blind optimisation, however good the
optimiser.** Fixing it required making the *environment* capacity-aware — candidates,
distances, detour test and stretch accounting all in cost units — not just the exporters.

In [ ]:
# The stability claim and its counter-example, side by side, from the data.
print("Seed spread (sd across 10 seeds), analytical offered load")
print(f"{'cell':22}{'single':>10}{'MARL':>10}   verdict")
print("-" * 62)
for c in CELLS:
    r = off.get(c, {})
    if "single" in r and "marlh32" in r:
        s, m = r["single"]["sd"], r["marlh32"]["sd"]
        verdict = "MARL tighter" if m < s else "SINGLE tighter  <- counter-example"
        print(f"{c:22}{s:>10.1f}{m:>10.1f}   {verdict}")

---
# 14. Limitations

Stated plainly, in rough order of how much they constrain the conclusion.

1. **Static traffic within an episode.** The project premise says "dynamic traffic", but each
   ns-3 run holds **one** matrix fixed. What is demonstrated is generalisation *across*
   matrices, not adaptation *within* an episode. Adaptation to a mid-episode traffic shift is
   untested.
2. **Three evaluation networks.** Only Abilene, GÉANT and Germany50 have public measured
   traffic. Nine matrices per regime is a small $n$; the Wilcoxon floor of $p = 0.004$ is a
   property of that $n$, not of a strong effect.
3. **One heterogeneous-capacity network, and it is the failure case.** GÉANT and Germany50
   are uniform 40 G. The claim "MARL adapts to unseen backbones" is therefore **not**
   demonstrated for unseen *capacity profiles*.
4. **Budget matched on steps, not on work.** ~6× more demands routed by the centralised arm.
   Favours the baseline, so it does not manufacture the result — but it is unmatched.
5. **Reward-form asymmetry only half-tested.** The centralised arm was re-run under the
   decentralised reward form. The reverse experiment was not done.
6. **Not deployable as-is.** Per-flow source-routed paths need MPLS or segment routing;
   destination-based IP forwarding cannot express them.
7. **No optimality ceiling.** The LP optimum was deliberately skipped (the question is "MARL
   vs deployed routing", and OSPF/ECMP are what is deployed). The greedy path-level
   reference in §10 is the only upper bound offered, and it is not tight.
8. **Training reward is an analytical surrogate.** Loss and delay are *measured* afterwards
   but never optimised directly, so the policy is not trained on the metric it is judged by
   at the packet level.
9. **Link failures untested.** The environment supports `fail_links`; every reported run uses
   0. Robustness to a failed link is unmeasured.
10. **Convergence and predictability not evaluated.** OSPF's real advantages — bounded
    convergence time, operator-auditable behaviour — are not measured here at all, and a
    learned policy is worse on both by construction.

---
# 15. Conclusions

**A single decentralised policy, trained once on 17 topologies, transfers zero-shot to
unseen backbones and reduces the bottleneck substantially on the two larger ones.** On GÉANT
it pulls a 145 % overload back under capacity (97.7 %) and cuts packet loss from 14.13 % to
3.36 %; on Germany50 it achieves the lowest loss of any method in overload (10.63 % against
OSPF's 27.29 %). In the feasible regime it buys ~30 points of headroom on GÉANT at strictly
better loss and delay than OSPF. Both effects are statistically significant when paired on
the matrix.

**It does so with ~15× fewer parameters than the centralised controller it is compared
against, and with far tighter seed-to-seed behaviour** under identical PPO hyperparameters —
so the stability is a property of the architecture, not of tuning.

**It fails on Abilene, and the reason is understood.** At 12 nodes with one slow link, a
correctly configured OSPF already makes the only decision the graph offers. Learned routing
needs routing freedom, and scale is what creates it.

**Three things must be said alongside the win.** Real equal-cost ECMP is a much stronger
baseline than initially reported and beats OSPF in both overload cells. In the feasible
regime nothing meaningfully separates the methods on delivered quality of service —
shortest-path routing is already sufficient when the network is not congested. And a
clairvoyant path-level greedy still beats the learned policy at 12 and 22 nodes, though not
at 50.

### Where this should go next, ranked

1. **Link-failure robustness.** The environment already supports it; evaluate the *saved*
   policies zero-shot with one link down. No retraining, ~2 h. Highest claim per hour.
2. **Within-episode traffic shifts.** Closes limitation 1, which is the gap between the
   project's stated premise and what it demonstrates.
3. **Per-topology specialists at the same budget.** Quantifies the generalisation gap — the
   ROAR-style comparison the literature does not report. A reference arm, not a replacement
   for the 17-topology design.
4. **A heterogeneous-capacity evaluation network.** The one thing that would let claim 3 in
   §14 be made rather than disclaimed.
5. **Real OSPF in ns-3** (`vendor/ns3-ospf`, ported and working). Buys control-plane realism;
   in a static evaluation it converges to the same paths, so this is citability, not
   correctness.

---
# Appendix A — Reproducing everything from scratch

```bash
# 0. environment
cd ~/thesis && source ns3ai-venv/bin/activate        # Python 3.9
pip install -r requirements_freeze.txt
bash configure_ns3.sh && cd ns-3-dev && ./ns3 build -j8    # a bare ./ns3 configure WILL fail

# 1. data (committed, regenerate only if needed)
python sndlib_to_json.py                              # SNDlib native -> topologies/*_sndlib.json

# 2. training — 20 runs, ~30 h wall clock at 14 concurrent
export OMP_NUM_THREADS=1 MKL_NUM_THREADS=1 OPENBLAS_NUM_THREADS=1
bash train_seeds10.sh

# 3. packet-level evaluation — 558 ns-3 simulations
NS3_TIMEOUT=3600 bash eval_seeds10.sh

# 4. aggregation
python rebuild_ns3_grid.py        # -> results/final_ns3_grid.json
python fill_offered_grid.py       # -> results/offered_grid.json
python decompose_reward.py        # -> results/reward_decomp.json
python select_width.py            # -> results/width_selection.json
python eval_greedy.py             # -> results/greedy_k3_grid.json
python significance_test.py       # paired Wilcoxon
python make_figures.py --out paper/tmlr-thesis/images
```

**Gotchas that cost real time:**

- `OMP_NUM_THREADS=1` when running many torch jobs — 24 uncapped processes took a 128-thread
  machine to load 275 and died with `MemoryError` + segfaults.
- `NS3_TIMEOUT=3600` for Germany50 (~17 min/sim); the 900 s default silently truncates it.
- `./ns3 run --no-build` in the eval launcher is load-bearing: without it every one of 16
  concurrent jobs spawns its own LTO link step and exhausts memory.
- `ns-3-dev/` is gitignored and has no git of its own. The scenario patches live in
  `ns3_patches/` and must be re-applied after any ns-3 reinstall — a known reproducibility
  gap.

# Appendix B — Artefact index

| Artefact | Contents |
|---|---|
| `results/final_ns3_grid.json` | **the** packet-level grid: loss, delay, utilisation × 6 cells × 4 arms × 10 seeds |
| `results/offered_grid.json` | analytical bottleneck offered load, same cells |
| `results/reward_decomp.json` | congestion/delay decomposition of the achieved return |
| `results/width_selection.json` | h=32 vs h=64 on held-out training topologies |
| `results/greedy_k3_grid.json` | path-level greedy reference |
| `results/ns3f_*/` | 144 export directories, per-simulation FlowMonitor state |
| `logs/train_marlh32cm_s*.log` | training traces behind the convergence figure |
| `paper/tmlr-thesis/` | the write-up (`main.pdf`) |

**Superseded — do not quote:** `results/matched_ns3_grid.json`, any `results/ns3m_*` or
`results/ns3f_*_abilenew_*` directory (capacity-blind learned paths and equal-hop ECMP),
and the `_singleH64gcap` arm (superseded reward form, retained in the grid as
`single_whole_ablation`).